# CLIP Zero-Shot Image Classification

学習済みCLIPを使用して、アップロードした画像と候補ラベルの類似度を比較する画像分類デモです。

## 実行方法

1. このノートブックをGoogle Colabで開きます。
2. セルを上から順に実行します。
3. 分類したい画像を1枚アップロードします。

## 注意事項

- このノートブックでは追加学習を行いません。
- 予測結果は、指定された候補ラベルの中から選ばれます。
- 候補に正解が含まれていなくても、いずれかのラベルが選ばれます。
- 表示スコアは候補間の相対的な値であり、正解である確率を保証するものではありません。
- 初回実行時には、学習済みモデルのダウンロードが必要です。


## 1. 依存ライブラリのインストール

実行環境に必要なライブラリをインストールします。
このセルはバージョンを固定していないため、再現性が必要な場合は動作確認済みのバージョンを別途記録してください。


In [ ]:
%pip install torch transformers pillow


## 2. モデルの準備

公開されている学習済みモデルを読み込みます。
GPUが利用できる場合はGPUを使用し、それ以外の場合はCPUを使用します。


In [ ]:
from io import BytesIO

import torch
from PIL import Image, ImageOps, UnidentifiedImageError
from google.colab import files
from IPython.display import display
from transformers import CLIPModel, CLIPProcessor

MODEL_ID = "openai/clip-vit-base-patch32"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CLIPModel.from_pretrained(MODEL_ID).to(device)
model.eval()

processor = CLIPProcessor.from_pretrained(MODEL_ID)

print(f"Device: {device}")


## 3. 候補ラベルの設定

分類したいカテゴリを英語で指定します。
ラベルを変更した場合は、このセルを再実行してください。


In [ ]:
labels = [
    "car",
    "apple",
    "house",
    "cat",
    "dog",
]


## 4. 分類関数の定義

画像と候補ラベルを前処理し、類似度に基づくスコアを計算します。
推論時には勾配追跡を無効にします。


In [ ]:
def classify_image(image, candidate_labels):
    """Return candidate labels and relative scores in descending order."""
    if not candidate_labels:
        raise ValueError("候補ラベルを1つ以上指定してください。")

    if any(
        not isinstance(label, str) or not label.strip()
        for label in candidate_labels
    ):
        raise ValueError("候補ラベルには空でない文字列を指定してください。")

    normalized_labels = [label.strip() for label in candidate_labels]

    if len(set(normalized_labels)) != len(normalized_labels):
        raise ValueError("候補ラベルが重複しています。")

    inputs = processor(
        text=normalized_labels,
        images=image.convert("RGB"),
        return_tensors="pt",
        padding=True,
    )
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        outputs = model(**inputs)
        scores = outputs.logits_per_image.softmax(dim=1)[0]

    results = list(zip(normalized_labels, scores.cpu().tolist()))
    return sorted(results, key=lambda result: result[1], reverse=True)


## 5. 画像のアップロードと分類

画像を1枚選択してください。
別の画像を分類する場合は、このセルを再実行します。

公開前には、このセルを含むすべての出力を削除してください。
出力にはアップロードした画像やファイル名が含まれる場合があります。


In [ ]:
uploaded_files = files.upload()

if not uploaded_files:
    print("画像が選択されませんでした。セルを再実行してください。")
elif len(uploaded_files) != 1:
    print("画像は1枚だけ選択してください。セルを再実行してください。")
else:
    image_bytes = next(iter(uploaded_files.values()))

    try:
        with Image.open(BytesIO(image_bytes)) as source_image:
            image = ImageOps.exif_transpose(source_image).convert("RGB")
    except (UnidentifiedImageError, OSError, ValueError):
        print("画像を読み込めませんでした。有効な画像を選択してください。")
    else:
        results = classify_image(image, labels)

        preview = image.copy()
        preview.thumbnail((640, 640))
        display(preview)

        print(f"Predicted class: {results[0][0]}")
        print("Relative scores:")

        for label, score in results:
            print(f"  {label}: {score:.4%}")
